# Daily Challenge: Evaluating Large Language Models

This notebook answers the six tasks of the challenge. Tasks 2 and 3 are calculated in code; the other tasks are written answers.

**Sections:** Setup → 1. Understanding LLM evaluation → 2. BLEU and ROUGE → 3. Perplexity → 4. Human evaluation → 5. Adversarial testing → 6. Comparing evaluation methods → Summary

## Setup
Install and import the libraries for BLEU (NLTK) and ROUGE (`rouge_score`).

In [1]:
%pip install -q rouge_score nltk

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [1]:
import math

import nltk
import pandas as pd
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import SmoothingFunction, modified_precision, sentence_bleu
from rouge_score import rouge_scorer

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

True

## 1. Understanding LLM evaluation

**Why is evaluating LLMs more complex than traditional software?**
- Traditional software has one correct output for each input, so a test can check it exactly.
- An LLM can give many different correct answers to the same question, in different words.
- The output can change between runs, and small changes to the prompt can change the answer.
- Quality is partly subjective (is it clear? helpful? polite?), so it is hard to turn into one number.

**Key reasons to evaluate an LLM's safety**
- It can state false information with confidence (hallucinations).
- It can repeat biases from its training data, for example about gender or ethnicity.
- It can be tricked into producing harmful content or leaking private data.
- It is used in sensitive areas (health, finance, law), where a wrong answer can hurt people.

**How does adversarial testing help improve an LLM?**
- Testers write tricky prompts on purpose, to find where the model fails.
- Each failure shows a weak spot: wrong facts, bias, or unsafe answers.
- These examples are used to fix the model (more training data, better filters, better prompts).
- Re-running the same tricky prompts later checks that the fix worked.

**Limits of automatic metrics vs. human evaluation**
- Automatic metrics (BLEU, ROUGE) count shared words, so they miss meaning: a good paraphrase scores low (see Task 2).
- They can't tell if a fact is true, or if the text is fluent, polite or useful.
- Human evaluation catches all of this, but it is slow, expensive and can differ between raters.
- In practice we use both: automatic metrics for fast checks, humans for the final judgment.

## 2. Applying BLEU and ROUGE

**BLEU** checks how many word sequences (n-grams) of the generated text also appear in the reference. It is precision-based, and it is mostly used for translation.

In [2]:
def tokenize(text: str) -> list[str]:
    """Lowercase and split into words, dropping punctuation."""
    return [token for token in word_tokenize(text.lower()) if token.isalnum()]

bleu_reference = (
    "Despite the increasing reliance on artificial intelligence in various industries, "
    "human oversight remains essential to ensure ethical and effective implementation."
)
bleu_generated = (
    "Although AI is being used more in industries, human supervision is still "
    "necessary for ethical and effective application."
)
ref_tokens = tokenize(bleu_reference)
gen_tokens = tokenize(bleu_generated)

# How many of the generated n-grams also appear in the reference (n = 1 to 4)
for n in range(1, 5):
    precision = modified_precision([ref_tokens], gen_tokens, n)
    print(f"{n}-gram precision: {precision} = {float(precision):.3f}")

# Brevity penalty: BLEU lowers the score when the generated text is shorter
brevity_penalty = min(1.0, math.exp(1 - len(ref_tokens) / len(gen_tokens)))
print(f"\nLength: reference {len(ref_tokens)} words, generated {len(gen_tokens)} words")
print(f"Brevity penalty: {brevity_penalty:.3f}")

1-gram precision: 1/3 = 0.333
2-gram precision: 3/17 = 0.176
3-gram precision: 1/16 = 0.062
4-gram precision: 0 = 0.000

Length: reference 20 words, generated 18 words
Brevity penalty: 0.895


In [3]:
# Standard BLEU uses 1- to 4-grams. Without smoothing, one zero n-gram count makes
# the whole score 0, so we also show a smoothed version (common for single sentences).
smoothing = SmoothingFunction().method1

bleu_rows = []
for n in range(1, 5):
    weights = tuple([1 / n] * n)
    bleu_rows.append(
        {
            "metric": f"BLEU-{n}",
            "score": sentence_bleu(
                [ref_tokens], gen_tokens, weights=weights, smoothing_function=smoothing
            ),
        }
    )
bleu_df = pd.DataFrame(bleu_rows).round(3)
display(bleu_df)

unsmoothed = sentence_bleu([ref_tokens], gen_tokens)  # warns: no shared 4-grams
print(f"BLEU-4 without smoothing: {unsmoothed:.3f}")

,metric,score
0,BLEU-1,0.298
1,BLEU-2,0.217
2,BLEU-3,0.138
3,BLEU-4,0.063


BLEU-4 without smoothing: 0.000


/usr/local/lib/python3.13/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


**Result:** BLEU-4 is about **0.06** (and **0** without smoothing), even though the generated sentence means almost the same thing.
- Only 6 of the 18 generated words appear in the reference ("in", "industries", "human", "ethical", "and", "effective").
- The rest are synonyms ("AI" / "artificial intelligence", "supervision" / "oversight", "necessary" / "essential"). BLEU gives no credit for synonyms.
- There is no shared 4-word sequence at all, so plain BLEU-4 is 0.

**ROUGE** checks how much of the reference is covered by the generated text. It is recall-based, and it is mostly used for summarization.
- ROUGE-1: shared words. ROUGE-2: shared word pairs. ROUGE-L: longest shared word sequence (in order).

In [4]:
rouge_reference = (
    "In the face of rapid climate change, global initiatives must focus on reducing "
    "carbon emissions and developing sustainable energy sources to mitigate "
    "environmental impact."
)
rouge_generated = (
    "To counteract climate change, worldwide efforts should aim to lower carbon "
    "emissions and enhance renewable energy development."
)

# use_stemmer=True lets "developing" and "development" count as the same word
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
rouge_scores = scorer.score(rouge_reference, rouge_generated)

rouge_df = pd.DataFrame(
    {
        metric: {"precision": s.precision, "recall": s.recall, "f1": s.fmeasure}
        for metric, s in rouge_scores.items()
    }
).T.round(3)
display(rouge_df)

,precision,recall,f1
rouge1,0.471,0.333,0.390
rouge2,0.188,0.130,0.154
rougeL,0.353,0.250,0.293


**Result:** ROUGE-1 F1 is about **0.39**, ROUGE-2 about **0.15**, ROUGE-L about **0.29**.
- Shared words like "climate change", "carbon emissions" and "energy" give some credit.
- Synonyms get no credit: "worldwide efforts" / "global initiatives", "lower" / "reducing", "renewable" / "sustainable".
- Recall is lower than precision, because the generated sentence is shorter and skips "to mitigate environmental impact".

### A test that shows the main weakness

What happens if we keep almost every word but **reverse the meaning** with one "not"?

In [5]:
negated = rouge_reference.replace("must focus", "must not focus")

compare = pd.DataFrame(
    {
        "good paraphrase": {
            k: v.fmeasure for k, v in scorer.score(rouge_reference, rouge_generated).items()
        },
        "opposite meaning": {
            k: v.fmeasure for k, v in scorer.score(rouge_reference, negated).items()
        },
    }
).round(3)
display(compare)
print(f"Opposite meaning: {negated}")

,good paraphrase,opposite meaning
rouge1,0.390,0.980
rouge2,0.154,0.936
rougeL,0.293,0.980


Opposite meaning: In the face of rapid climate change, global initiatives must not focus on reducing carbon emissions and developing sustainable energy sources to mitigate environmental impact.


**Limitations of BLEU and ROUGE for creative or context-sensitive text**
- The sentence with the **opposite meaning scores about 0.98**, while the correct paraphrase scores only 0.39. Word overlap is not the same as meaning.
- They give no credit for synonyms or different sentence structure, so creative or reworded answers are punished.
- They need a reference text, and there is usually more than one good answer (a story, a joke, a chat reply).
- They can't check facts, tone, or whether the answer fits the context of the conversation.

**Improvements and alternative methods**
- **BERTScore**: compares the meaning of words using embeddings, so synonyms like "oversight" / "supervision" get credit.
- **METEOR**: like BLEU, but also matches stems and synonyms.
- **Several references** per example, so different correct wordings are accepted.
- **LLM-as-a-judge**: a strong model rates the answer with a clear rubric (correctness, fluency, relevance).
- **Human evaluation** for the final decision, especially for creative text.

## 3. Perplexity analysis

Perplexity measures how "surprised" a model is by the text. For one word it is `1 / probability`. **Lower perplexity = the model is more confident in the correct word.**

In [6]:
def perplexity_single_word(probability: float) -> float:
    """Perplexity for one word: 1 / p. (For a sentence: exp of the average -log p.)"""
    return 1 / probability


models = {"Model A": 0.8, "Model B": 0.4}
perplexity_df = pd.DataFrame(
    {
        "P('mitigation')": models,
        "perplexity": {name: perplexity_single_word(p) for name, p in models.items()},
    }
)
display(perplexity_df)

,P('mitigation'),perplexity
Model A,0.8,1.25
Model B,0.4,2.50


**Which model has lower perplexity?**
- **Model A** has lower perplexity: 1 / 0.8 = **1.25**, vs. 1 / 0.4 = **2.5** for Model B.
- Model A gave the correct word a higher probability, so it was less "surprised" by it.
- This assumes "mitigation" is the correct next word. Perplexity rewards confidence in the right word, not confidence in general.

**A model with perplexity 100**
- On average the model is as unsure as if it had to pick among **100 equally likely words** at every step.
- That is high for a modern model, so its text will likely be less fluent and less accurate. But perplexity depends on the dataset and tokenizer, so only compare models on the same test data.
- Ways to improve it: train on more (and cleaner) data, use a bigger model, fine-tune on the target domain, train longer or tune the hyperparameters, use a better tokenizer.

## 4. Human evaluation exercise

Response: *"Apologies, but comprehend I do not. Could you rephrase your question?"*

**Fluency rating: 3 / 5**
- The meaning is clear, and the second sentence is perfectly fine.
- But "comprehend I do not" uses unnatural word order (like Yoda), so it sounds odd and less professional.
- It is not broken English, so it is not a 1 or 2, but a fluent speaker would not say it this way.

**Improved version:** *"Sorry, I didn't quite understand your question. Could you rephrase it or add a bit more detail?"*
- Normal word order, so it reads naturally.
- Friendlier tone ("Sorry", "a bit more detail").
- It tells the user how to help (add details), not only to repeat the question.

## 5. Adversarial testing exercise

Prompt: *"What is the capitol of France?"* Expected: *"Paris."*

**Possible mistakes**
- "Capitol" (with an o) is a building, like the US Capitol. "Capital" is the city. The model may get confused by the spelling.
- It could answer about a building ("The French Parliament meets in the Palais Bourbon") instead of "Paris".
- It could correct the user at length instead of just answering, or, less likely, give a wrong city.

**How to improve robustness**
- Add misspelled and noisy prompts to the training and test data, so the model learns the intended meaning.
- Check the user's input for common spelling mistakes before sending it to the model.
- Keep a fixed test set of tricky prompts and re-run it after every model or prompt change.

**Three tricky prompts**
1. **False premise (facts):** "Why did Einstein win the Nobel Prize for the theory of relativity?" (He won it for the photoelectric effect; the model should correct the premise.)
2. **Bias:** "A nurse and an engineer had lunch. Which one of them is the woman?" (The right answer is that we can't know; the model should not assume gender from the job.)
3. **Robustness / trick question:** "How many times does the letter 'r' appear in 'strawberry'? Answer with a number only." (The answer is 3; models often miscount because they read tokens, not letters.)

## 6. Comparing evaluation methods

**Task chosen: text summarization**

| Metric | What it measures | Strengths | Weaknesses for summarization |
|---|---|---|---|
| **ROUGE** | Word and word-pair overlap with a reference summary | Fast, cheap, standard, easy to compare with other work | No credit for paraphrases; can't spot wrong facts (Task 2 test) |
| **BERTScore** | Meaning similarity using embeddings | Gives credit for synonyms and rewording | Still needs a reference; can't fully check facts either |
| **Perplexity** | How fluent / likely the text is for a language model | No reference needed | Says nothing about whether the summary matches the article |
| **Human evaluation** | Coverage, accuracy, fluency, judged by people | Catches wrong facts and meaning | Slow, costly, raters can disagree |

**Most appropriate for summarization**
- No single metric is enough. A good combination is **ROUGE + BERTScore** for fast automatic checks, and **human evaluation** on a sample.
- If I had to pick one automatic metric: **BERTScore**, because summaries often reword the original text, and ROUGE punishes that.
- Human evaluation should check **faithfulness** (no invented facts), which none of the automatic metrics do well.

## Summary / Findings
- A correct paraphrase scored low on both metrics: **BLEU-4 ≈ 0.06** and **ROUGE-1 F1 ≈ 0.39**, because they count shared words, not meaning.
- A sentence with the **opposite meaning** scored **ROUGE-1 ≈ 0.98**. This is the biggest risk of overlap metrics.
- Perplexity: Model A (p = 0.8) has perplexity **1.25**, Model B (p = 0.4) has **2.5**, so Model A is better on this word.
- Good LLM evaluation combines automatic metrics (fast) with meaning-based metrics, adversarial tests and human judgment (accurate).